# Supplementary material figures — S.1 to S.57

Companion to `paper_figures.ipynb`: regenerates every figure of the
supplementary document from the same certified cache (`reactor.paper`),
via `reactor.paper.figures_si`. Figure numbering and layout mirror
the supplementary document; the data is the updated
certified dataset.

Everything here reads from the cache — run the sweeps first
(`python -m scripts.run_paper_sweep`), then this notebook re-renders in
about two minutes. Output lands in
`results/paper/<RESOLUTION>/figures/si/`.

## Run on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-engineering/ammonia_synthesis_reactor/blob/main/notebooks/si_figures.ipynb)

The next cell is a **no-op on a local checkout**. On Colab it installs this
package (pulling `pymrm` from PyPI), downloads the archived dataset — its
DOI and download URL live in `reactor.archive`, filled in at deposit
time — verifies it against its SHA-256 manifest and unpacks it into the local cache. Every
cell below then re-renders from cache, exactly as on a machine that ran
the sweeps.

In [ ]:
# ── Google Colab bootstrap (no-op on a local checkout) ───────────────────────
import os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    os.environ["REACTOR_RESOLUTION"] = "publication"   # the archived tier
    if not os.path.exists("data/inputs"):
        !git clone --depth 1 https://github.com/computational-chemical-engineering/ammonia_synthesis_reactor.git
        %cd ammonia_synthesis_reactor
    %pip install -q .
    # The archive identifiers live in one place, reactor.archive, so a
    # deposit is filled in once and every consumer follows.
    from reactor import archive
    if not os.path.exists("dataset/manifest.json"):
        DATASET_ZIP_URL = archive.dataset_zip_url()   # raises while pending
        !wget -q -O _dataset.zip "$DATASET_ZIP_URL"
        !unzip -q -o _dataset.zip
    from reactor.paper import dataset
    check = dataset.verify()                 # checksums vs manifest.json
    assert check["ok"], check
    print(f"dataset verifies: {check['n_checked']} files")
    print(dataset.restore_cache())           # populate the cache; never overwrites

In [ ]:
import os

RESOLUTION = os.environ.get("REACTOR_RESOLUTION", "publication")   # "draft" | "publication"

import pandas as pd

from reactor.paper import (cases, dimensionless, figures, figures_si,
                           provenance, runner, settings, validation)

case_table = cases.load_case_table()
print(provenance.describe())

## S.1 — NH₃ diffusivity correlation vs Chapman et al.

The correlation curves come from `GasMixtureCorrelations`; the experimental
points are the co-author's digitized Chapman et al. data
(`settings.S1_DIFFUSION_DATA_CSV`, provenance in the CSV header). The data
is the NH₃–H₂ *binary* diffusivity at the source-figure conditions (~1 atm /
~273 K), which the pipeline's Chapman–Enskog machinery reproduces to <1%;
the panels therefore compare at those conditions and at trace NH₃ in H₂.

In [ ]:
fig = figures_si.figure_s1_diffusivity(settings.S1_DIFFUSION_DATA_CSV)
figures_si.save_si(fig, "figS01_diffusivity", RESOLUTION)

## S.2–S.5 — Radial Péclet / diffusive Damköhler scaling and regime maps

Per-case scalars from `reactor.paper.dimensionless` (means and maxima
over the membrane-active length, z > 0.05 m; injection of the cached 2D
fields into `reactor.postprocessing.compute_dimensionless_numbers`).

In [ ]:
scalars = dimensionless.sweep_scalars(case_table, RESOLUTION)
print(f"{len(scalars)} cases")
scalars.head(3)

In [ ]:
fig = figures_si.figure_s2_pe_da_whsv(scalars)
figures_si.save_si(fig, "figS02_pe_da_whsv", RESOLUTION)

In [ ]:
fig = figures_si.figure_s3_pe_da_radius(scalars)
figures_si.save_si(fig, "figS03_pe_da_radius", RESOLUTION)

In [ ]:
fig = figures_si.figure_s4_pe_ratio(scalars)
figures_si.save_si(fig, "figS04_pe_ratio", RESOLUTION)

In [ ]:
fig = figures_si.figure_s5_regime_map(scalars)
figures_si.save_si(fig, "figS05_regime_map", RESOLUTION)

## S.6 — Rossetti validation, remaining conditions

The conditions not shown in main-text Figure 3, same panel style. Reads
the cached validation table (`reactor.paper.validation.run_rossetti`).

In [ ]:
rossetti = validation.run_rossetti(model="2d")
fig = figures.figure_s6_rossetti_rest(rossetti)
figures_si.save_si(fig, "figS06_rossetti_rest", RESOLUTION)

## S.7 — 1D vs 2D across temperature (G5 / G6)

The temperature-sweep companion to main-text Figures 6–8. Both 598 K
cases follow deferred decision D3 and are excluded from the series by
default.

In [ ]:
summary_2d = runner.load_summary(RESOLUTION, settings.MODEL_2D)
summary_1d = runner.load_summary(RESOLUTION, settings.MODEL_1D)
mask = ~summary_2d["Case_ID"].isin(settings.CASES_598K) if settings.INCLUDE_598K == "exclude" else slice(None)
fig = figures_si.figure_s7_temperature(summary_2d[mask], summary_1d)
figures_si.save_si(fig, "figS07_1d2d_temperature", RESOLUTION)

## S.8–S.55 — Per-case 2D fields

One figure per case — NH₃ mole fraction (a) and temperature (b) — in the
SI document's order (G1/G2 interleaved per GHSV, then G3/G4, G5/G6, G7,
G8). The figure-number-to-case map is written next to the images
(`si_field_map.csv`).

In [ ]:
mapping = figures_si.render_si_fields(RESOLUTION, case_table, verbose=0)
print(mapping["status"].value_counts().to_dict())
mapping.head(8)

## S.56 — CP profiles across temperature (G5 / G6)

In [ ]:
fig = figures_si.figure_s56_cp_temperature(RESOLUTION)
figures_si.save_si(fig, "figS56_cp_temperature", RESOLUTION)

## S.57 — parity of the three 1D variants

Companion to main-text Section 3.3.3 (Table 2): plain 1D (CP = 1),
$Sh_{opt}(\kappa)$-corrected and mechanistic-corrected 1D against the
certified 2D KPIs, for H₂ conversion and NH₃ recovery.

In [ ]:
s2 = pd.read_csv(settings.summary_csv(RESOLUTION, settings.MODEL_2D))
s1 = pd.read_csv(settings.summary_csv(RESOLUTION, settings.MODEL_1D))
sc = pd.read_csv(settings.summary_csv(RESOLUTION, settings.MODEL_1D_CORRECTED))
ss = pd.read_csv(settings.summary_csv(RESOLUTION, settings.MODEL_1D_SCREENED))
fig = figures_si.figure_s57_parity(s2, s1, sc, ss)
figures_si.save_si(fig, "figS57_parity", RESOLUTION)